### **Import libraries**: 
- **Defeat Beta API** is an open-source alternative to Yahoo Finance's market data APIs. Moreover, it has the added ability of permitting news retrieval.
- **Hugging Face** is a platform and library for utilizing machine learning models.
- **InferenceClient** is a library available on Hugging Face and is primarily used to perform inference – the process of reaching conclusions given a set of inputs.
- **Transformers** is a Python library created by Hugging Face that allows one to download, manipulate, and run thousands of pretrained, open-source AI models.
- **Pipeline** is a package within Transformers that simplifies the process of working with advanced AI models.
- **UUID** is a library that provides a way to generate universally unique identifiers. In this script, it is used to create identifiers for headlines.

In [116]:
import requests
import defeatbeta_api
from defeatbeta_api.data.ticker import Ticker
from defeatbeta_api.data.tickers import Tickers
from huggingface_hub import InferenceClient
from transformers import pipeline
import uuid
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import sys

### **Create a data frame with ticker, headlineID, headline, article publication date, and article source columns.** ###

In [117]:
def package(ticker_lst_param):
    df_main = pd.DataFrame(columns = ["Ticker", "HeadlineID", "Headline", "Timestamp", "Source"])
    #df_main.set_index(["Ticker"], inplace = True)
    #df_main.sort_index(inplace = True)
    for element in ticker_lst_param:
        tickers = [element["Ticker"] for i in range(len(element["Headline"]))]
        headline_IDs = [str(uuid.uuid4())[:8] for i in range(len(element["Headline"]))]
        data = {"Ticker" : tickers, "HeadlineID" : headline_IDs, "Headline" : element["Headline"], "Timestamp" : element["Timestamp"], "Source" : element["Source"]}
        df = pd.DataFrame(data = data)
        #df.set_index(["Ticker"], inplace = True)
        #df.sort_index(inplace = True)
        df_main = pd.concat([df_main, df])
    #df_main.sort_index(inplace = True)
    return df_main

### **Use the Defeat Beta API to pull news associated with each ticker.**

### **To modify the tickers used, make changes to tickers_file.txt**

In [118]:
def get_data(tickers_file):
    with open(tickers_file, "r") as f:
        tickers = f.readline()
        tickers = tickers[1:-1]
        tickers = tickers.replace('"', "")
        tickers = tickers.split(",")
        tickers = [element.strip() for element in tickers]
        tickers_cl = Tickers(tickers)
    news = tickers_cl.news()
    headline_data = [{"Ticker" : element, "Headline" : news[element].get_news_list()["title"], "Timestamp" : news[element].get_news_list()["report_date"], "Source" : news[element].get_news_list()["publisher"]} for element in tickers]
    headline_data_packaged = package(headline_data)
    return headline_data_packaged, tickers

### **Utilize FinBERT - a pre-trained NLP model specialized for financial sentiment analysis – adapted by ProsusAI to analyze the probabilities (positive/negative/neutral) associated with each headline.**

In [119]:
def add_probabilities(ticker_news_data_df_param, api_key_param):
    pipe = pipeline("text-classification", model="ProsusAI/finbert")
    client = InferenceClient(provider="hf-inference", api_key=api_key_param)
    ticker_news_data_df_param[["pos", "neu", "neg"]] = 0.0, 0.0, 0.0
    ticker_news_data_df_param["Timestamp"] = pd.to_datetime(ticker_news_data_df_param["Timestamp"])
    ticker_news_data_df_param = ticker_news_data_df_param[ticker_news_data_df_param["Timestamp"] >= (datetime.today() - timedelta(days=3))]
    #ticker_news_data_df_param = ticker_news_data_df_param.head(15)
    for index, row in ticker_news_data_df_param.iterrows():
        headline = row["Headline"]
        result = client.text_classification(headline, model="ProsusAI/finbert")
        for probability in result:
            row = row.copy()
            match probability["label"]:
                case "positive":
                    ticker_news_data_df_param.loc[index, "pos"] = probability.score
                case "neutral":
                    ticker_news_data_df_param.loc[index, "neu"] = probability.score
                case "negative":
                    ticker_news_data_df_param.loc[index, "neg"] = probability.score

    #ticker_news_data_df_param = pd.pivot_table(ticker_news_data_df_param, index = ["Ticker", "HeadlineID"], aggfunc = str())
    ticker_news_data_df_param.set_index(["Ticker", "Timestamp", "HeadlineID"], inplace = True)
    ticker_news_data_df_param.sort_index(inplace = True, level = 0)
    print(ticker_news_data_df_param)
    #ticker_news_data_df_param.sort_index(inplace = True)
    #print(ticker_news_data_df_param.index)
    #print(ticker_news_data_df_param.loc["2026-04-04", "TSLA"])
    return ticker_news_data_df_param

### **Main program used to collectively execute the whole script and retrieve the API key used for the inference package.**

In [120]:
def main():
    with open("inference_api_key.txt", "r") as f:
        api_key = f.readline().strip()
    ticker_news_data_df, tickers = get_data("tickers.txt")
    ticker_news_data_df_with_probabilities = add_probabilities(ticker_news_data_df, api_key)
main()

2026-04-06 08:02:38 INFO httpx MainThread - HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-06 08:02:38 INFO httpx MainThread - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-04-06 08:02:38 INFO httpx MainThread - HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-06 08:02:38 INFO httpx Thread-auto_conversion - HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert "HTTP/1.1 200 OK"
2026-04-06 08:02:38 INFO httpx Thread-auto_conversion - HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/main "HTTP/1.1 200 OK"
2026-04-06 08:02:38 INFO httpx MainThread - HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-06 08:02:38 INFO httpx Thread-auto_conversion - HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/discussions?p=0 "HTTP/1.1 200 OK"
2026-04-06 08:02:38 INFO httpx MainThread - HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-04-06 08:02:38 INFO httpx Thread-auto_conversion - HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/refs%2Fpr%2F29 "HTTP/1.1 200 OK"
2026